# Week 8 — An agent with tools over the course graph

The lecture defined an **agent**: a system in which the model decides the next step after
seeing the result of the last one, and acts through **tools**. In this session you build
one, a step at a time. LangChain runs the loop. Your work is what a framework leaves to
the engineer: the tools, their descriptions, what they return, which of them may write,
how their errors reach the model, and when the loop stops.

| Part | What you do |
|---|---|
| 1 | LangChain in brief: connect a model, send one message |
| 2 | Step 1: one tool |
| 3 | Step 2: a second tool, and the loop |
| 4 | Step 3: the course retrievers as tools |
| 5 | Step 4: a fixed query over the graph |
| 6 | Step 5: memory as tools |
| 7 | Step 6: errors and a step limit |
| 8 | Tools for your own project |

You write the tools in this notebook. `week8/agent_kit.py` connects the model and prints
what the agent did; read it when Part 1 first uses it.

**Install.** This week adds LangChain. From the repository root, with your environment
active, run `pip install -r requirements.txt`, then restart the kernel.

**Backend.** As in Week 7, the model runs on **OpenRouter**: `OPENROUTER_API_KEY` must be
in `.env` at the repository root. Free models allow **50 requests a day** per account.
One pass through this notebook makes about 30, and running a cell again makes its
requests again. The retrievers and the memory still embed with your Gemini key, and the
exchange-rate tool needs an internet connection.

In [23]:
# Run this notebook from any folder. Week 8's helpers live in week8/; the tools reuse the
# Week 6 retrievers, the Week 7 memory store, and the modules of Weeks 2 to 5 they need.
import sys, pathlib
_root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (p / "requirements.txt").exists()), pathlib.Path.cwd().parent)
for _d in ("common", "week2", "week3", "week4", "week5", "week6", "week7", "week8"):
    sys.path.insert(0, str(_root / _d))
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

---
## Setup — the course graph from Week 6

This notebook reads the graph that Week 6 built and does **not** rebuild it: rebuilding
empties the database, and your Lab 2 graph with it. If the check below fails, run
`python week6/build_graph.py` from the repository root, then re-run your Lab 2
`ingest.py`.

Neo4j must be running: start Docker Desktop first, then `docker compose up -d` from the
repository root.

In [24]:
from embedding_client import EmbeddingClient
from graph_db import connect
from retrievers import read
from vector_check import require_same_model

driver = connect()
emb = EmbeddingClient()
require_same_model(driver, emb)
print(read(driver, "MATCH (e:Entity) RETURN count(e) AS entities")[0],
      read(driver, "MATCH (c:Chunk) RETURN count(c) AS chunks")[0])

{'entities': 372} {'chunks': 183}


You should see 372 entities and 183 chunks, and no error from the embedding-model check.

---
## Part 1 — LangChain in brief

**LangChain** is a library of components for applications built on models. This notebook
uses five of them:

| Component | What it is | In this notebook | Documentation |
|---|---|---|---|
| Chat model | one interface to many providers | `ChatOpenAI`, pointed at OpenRouter | [Models](https://docs.langchain.com/oss/python/langchain/models), [ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai) |
| Messages | the conversation, as messages with roles | what `ask` prints | [Messages](https://docs.langchain.com/oss/python/langchain/messages) |
| Tools | Python functions the model may ask to have run | `@tool`, from Part 2 on | [Tools](https://docs.langchain.com/oss/python/langchain/tools) |
| `create_agent` | a ready-made agent: the lecture's loop | from Part 2 on | [Agents](https://docs.langchain.com/oss/python/langchain/agents) |
| Middleware | code that runs around each model call or tool call, without changing the loop | the three below | [Overview](https://docs.langchain.com/oss/python/langchain/middleware/overview), [Prebuilt middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in) |

The middleware you will add:

| Middleware | What it does | Part |
|---|---|---|
| [`ToolCallLimitMiddleware`](https://docs.langchain.com/oss/python/langchain/middleware/built-in#tool-call-limit) | refuses calls to a tool beyond a set number, and tells the model so | 4 |
| [`ModelCallLimitMiddleware`](https://docs.langchain.com/oss/python/langchain/middleware/built-in#model-call-limit) | stops a run after a set number of model calls | 4 |
| [`ToolErrorMiddleware`](https://docs.langchain.com/oss/python/langchain/middleware/built-in#tool-error) | returns an exception raised inside a tool to the model, as the tool's result | 7 |

Since version 1, `create_agent` is built on
[**LangGraph**](https://docs.langchain.com/oss/python/langgraph/overview), LangChain's
library for running a model and tools as a graph of steps. Week 9 looks inside it.

The documentation describes LangChain's latest release; the course pins version 1.4.7 in
`requirements.txt`, which has everything used here. The ChatOpenAI page recommends
LangChain's own OpenRouter package, because `ChatOpenAI` drops the fields OpenAI does
not define, such as a model's reasoning. OpenRouter accepts OpenAI's format, and this
notebook needs none of those fields.

The framework runs the loop and turns your functions into tool definitions. It does not
decide which tools exist, how they are described, what they return, or what may be
written. That is your work in this notebook.

`openrouter_model()` in `agent_kit.py` returns LangChain's chat model on OpenRouter, with
the course's pinned free model. First, one message, with no tools and no agent:

In [25]:
from agent_kit import openrouter_model

model = openrouter_model()
reply = model.invoke("In one sentence: what is a software agent?")      # one request
print(type(reply).__name__)
print(reply.text)

AIMessage
A software agent is a computer program that acts autonomously on behalf of a user or another program to perceive its environment, make decisions, and take actions to achieve specific goals.


The reply is an `AIMessage`, LangChain's name for a message with the role `assistant`.
LangChain calls the four roles `system`, `human`, `ai` and `tool`, and translates them
into each provider's own names; for OpenRouter these are `system`, `user`, `assistant`
and `tool`.

---
## Part 2 — Step 1: one tool

A tool in LangChain is a Python function marked `@tool`. The framework builds the tool's
definition from the function itself:

| The model sees | Taken from |
|---|---|
| the name | the function's name |
| the description | its docstring |
| the parameters | its type hints, as a JSON schema |

The function below asks the National Bank of Ukraine's public API for the official
hryvnia rate of one currency. It needs no key.

In [26]:
import requests
from langchain.tools import tool

NBU = "https://bank.gov.ua/NBUStatService/v1/statdirectory/exchange"


@tool
def exchange_rate(currency: str) -> str:
    """The official hryvnia exchange rate set by the National Bank of Ukraine for one
    currency, given its ISO 4217 code such as EUR or USD. Use it for any question about
    a current rate."""
    rows = requests.get(f"{NBU}?valcode={currency}&json", timeout=10).json()
    if not rows:                                    # an unknown code returns an empty list
        return f"No NBU rate for {currency!r}."
    r = rows[0]
    return f"1 {r['cc']} = {r['rate']} UAH, set by the NBU for {r['exchangedate']}"

What the model will be told about this tool, exactly as it is sent:

In [27]:
from agent_kit import definition

print(definition(exchange_rate))

{
  "type": "function",
  "function": {
    "name": "exchange_rate",
    "description": "The official hryvnia exchange rate set by the National Bank of Ukraine for one\ncurrency, given its ISO 4217 code such as EUR or USD. Use it for any question about\na current rate.",
    "parameters": {
      "properties": {
        "currency": {
          "type": "string"
        }
      },
      "required": [
        "currency"
      ],
      "type": "object"
    }
  }
}


Compare it with the lecture's table of a tool's parts. The docstring has become the
**description**: writing the docstring is writing a prompt. The body of the function is
not sent, and the model never sees it.

A tool is still an ordinary function. Run it yourself, with no model involved:

In [28]:
print(exchange_rate.invoke({"currency": "EUR"}))

1 EUR = 50.5766 UAH, set by the NBU for 26.10.2026


Now the agent. `create_agent` takes the model, the tools and a system instruction.
`ask` in `agent_kit.py` sends one question, then prints the turn's steps one line each, in
the order they were exchanged, then the answer in full, and the number of model calls the
turn made.

In [29]:
from langchain.agents import create_agent
from agent_kit import ask

SYSTEM = "You are a helpful assistant. Use the tools when they help."

agent = create_agent(model, tools=[exchange_rate], system_prompt=SYSTEM)
history = ask(agent, "What is the hryvnia rate for the euro?")

user        What is the hryvnia rate for the euro?
assistant   calls exchange_rate {"currency": "EUR"}
tool        exchange_rate → 1 EUR = 50.5766 UAH, set by the NBU for 26.10.2026

──── answer ───────────────────────────────────────────────────────────
The official exchange rate set by the National Bank of Ukraine (NBU) is:

**1 EUR = 50.5766 UAH** (as of October 26, 2026)
───────────────────────────────────────────────────────────────────────
2 model calls in this turn, 29 since the kernel started


Read the lines as the lecture's single tool call: the question, the model's call, the
tool's result, and the answer written from it. There were two model calls: the first
chose the tool, the second read its result. Between them the application, here
LangChain, ran the function.

Now ask the model the same question **without** the tool, as in Part 1:

In [30]:
reply = model.invoke("What is the hryvnia rate for the euro?")      # one request, no tools
print(reply.text)

I do not have access to real-time financial data, so I cannot provide the **exact current exchange rate** for the Euro (EUR) to the Ukrainian Hryvnia (UAH).

For the most accurate and up-to-date rate, please check one of these reliable sources:

1.  **National Bank of Ukraine (NBU) – Official Rate:** [bank.gov.ua](https://bank.gov.ua/) (This is the official reference rate set daily).
2.  **Major Ukrainian Banks (Commercial Rates):**
    *   PrivatBank
    *   Monobank
    *   Oschadbank
    *   Raiffeisen Bank
    *   Sense Bank
    *   *(Check their websites or apps for "Buy/Sell" rates, which differ from the NBU rate).*
3.  **Financial Aggregators:**
    *   [Minfin.com.ua](https://minfin.com.ua/currency/euro/) (Excellent for comparing bank rates).
    *   [Finance.ua](https://finance.ua/ru/currency)
    *   Google Finance / XE.com / Investing.com.

**Important Note:** There is usually a difference between the **Official NBU Rate** (used for accounting, customs, and statistics) and t

**Exercise 1.** What did the model answer without the tool? If it gave a figure, how would
a user know whether it is current? Compare it with the agent's answer above, which states
the date the NBU set the rate for. Run the cell again: does the model answer the same way?

---
## Part 3 — Step 2: a second tool, and the loop

The calculator from the lecture. The type hint `Literal[...]` becomes an `enum` in the
schema, so the definition limits `op` to four values, as `Literal` did in Week 5.

In [31]:
import operator
from typing import Literal

OPS = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}


@tool
def calculate(a: float, op: Literal["+", "-", "*", "/"], b: float) -> float:
    """Apply one arithmetic operation to two numbers. Use it for every calculation
    instead of working it out yourself."""
    return OPS[op](a, b)


print(definition(calculate))

{
  "type": "function",
  "function": {
    "name": "calculate",
    "description": "Apply one arithmetic operation to two numbers. Use it for every calculation\ninstead of working it out yourself.",
    "parameters": {
      "properties": {
        "a": {
          "type": "number"
        },
        "op": {
          "enum": [
            "+",
            "-",
            "*",
            "/"
          ],
          "type": "string"
        },
        "b": {
          "type": "number"
        }
      },
      "required": [
        "a",
        "op",
        "b"
      ],
      "type": "object"
    }
  }
}


Give the agent both tools, and ask a question that needs both:

In [32]:
agent = create_agent(model, tools=[exchange_rate, calculate], system_prompt=SYSTEM)
history = ask(agent, "How many hryvnias is 250 euros at today's NBU rate?")

user        How many hryvnias is 250 euros at today's NBU rate?
assistant   calls exchange_rate {"currency": "EUR"}
tool        exchange_rate → 1 EUR = 50.5766 UAH, set by the NBU for 26.10.2026
assistant   calls calculate {"b": 50.5766, "a": 250, "op": "*"}
tool        calculate → 12644.15

──── answer ───────────────────────────────────────────────────────────
At today's NBU rate (26.10.2026), **1 EUR = 50.5766 UAH**.

**250 EUR × 50.5766 = 12,644.15 UAH**

So, **250 euros = 12,644.15 hryvnias** at the official National Bank of Ukraine rate.
───────────────────────────────────────────────────────────────────────
3 model calls in this turn, 33 since the kernel started


The model called one tool, read its result, and chose the next. This is the **agent
loop**: no code decided the order of the steps. The run ends when the model answers
instead of calling a tool.

Count the model calls in the turn. Each is one request, and each sends the whole history
again, tool results included.

**Exercise 2.** Delete the sentence *"Use it for every calculation instead of working it
out yourself"* from the docstring of `calculate`, and run this part's cells again. Does the
model still call the calculator? (About three requests.)

---
## Part 4 — Step 3: the course retrievers as tools

Week 6's retrievers are already functions: a question in, a list of sources out. A tool
wraps each with a description and one parameter, the query. Two decisions move from your
code to the model: **whether** to retrieve, and **with what query**.

An agent that searches can keep searching when the results do not contain the answer, and
every search is a request. Two limits are set from here on, both in the code: at most two
searches for one question, which the instruction also asks for, and at most six model
calls in one run. A search beyond the second is refused, and the model is told so. Part 7
explains both limits.

In [33]:
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware
from retrievers import by_name, hybrid


@tool
def facts_by_name(query: str) -> str:
    """Facts from the course graph about tools, topics and labs named in the query.
    Returns nothing if the query names none."""
    facts = by_name(driver, query)
    return "\n".join(facts) if facts else "No facts: the query names nothing in the graph."


@tool
def search_course(query: str) -> str:
    """Search the course materials (lectures, practicals and plans) for passages and
    facts relevant to the query. Use it for any question about the course."""
    return "\n".join(hybrid(driver, emb, query))


COURSE = ("You answer questions about the Modern AI Systems course. Use the tools to find "
          "the answer, and answer only from what they return. Search at most twice for "
          "one question; if the results do not contain the answer, say so.")

SEARCHES = ToolCallLimitMiddleware(tool_name="search_course", run_limit=2)   # Part 7 explains
LIMIT = ModelCallLimitMiddleware(run_limit=6, exit_behavior="end")

course_tools = [exchange_rate, calculate, facts_by_name, search_course]
agent = create_agent(model, tools=course_tools, system_prompt=COURSE,
                     middleware=[SEARCHES, LIMIT])
history = ask(agent, "What distance does Chroma use by default?")

user        What distance does Chroma use by default?
assistant   calls search_course {"query": "Chroma default distance metric"}
tool        search_course → this one to the taxonomy; the next slide unpacks HNSW, and the one after takes the middle row in detail. Give a decision heuristic rather than a ranking: start from the data platform the project is alread … (4,060 characters)
assistant   calls search_course {"query": "Chroma default distance cosine space hnsw"}
tool        search_course → notes: ≈4 min. Two points to land, and the second is the one that changes a student's design. First: capability does not follow from a data type. MySQL Community can store a vector and parse one from a st … (4,363 characters)

──── answer ───────────────────────────────────────────────────────────
According to the course materials (Week 3 lecture notes), **Chroma's unconfigured default
distance metric is squared Euclidean distance**, not cosine similarity. The course
explicitly notes this when di

A follow-up in the same conversation: pass `history` back to `ask`.

In [34]:
history = ask(agent, "How do we change it?", history)

user        How do we change it?
assistant   calls search_course {"query": "hnsw:space cosine chroma configuration vector_store.py"}
tool        search_course → this one to the taxonomy; the next slide unpacks HNSW, and the one after takes the middle row in detail. Give a decision heuristic rather than a ranking: start from the data platform the project is alread … (3,889 characters)
assistant   calls search_course {"query": "vector_store.py hnsw space cosine chroma"}
tool        search_course → notes: ≈4 min. Two points to land, and the second is the one that changes a student's design. First: capability does not follow from a data type. MySQL Community can store a vector and parse one from a st … (4,363 characters)
assistant   calls search_course {"query": "vector_store.py chroma collection create hnsw space cosine"}
tool        search_course → Tool call limit exceeded. Do not call 'search_course' again.

──── answer ───────────────────────────────────────────────────────────
Based o

Read the second turn's tool call. The follow-up names nothing, so the query the model
wrote shows how it read the conversation. It wrote the query with the whole history in
view: the step that Week 7's rewrite performed now happens inside the tool call, and it
can go wrong in the same way. A system should log every query the model writes, as Week 7
logged the rewrite.

Count the searches. The course materials say that `hnsw:space` must be set to cosine, but
never show the line of code that sets it, and a model looking for that line can keep
searching. Without a limit on searches, it searched until the limit on model calls
stopped the run, unanswered; the instruction to search at most twice did not stop it. If
a third search was refused here, the model then answered from what it had. Compare the
answer with what the searches returned: code in the answer did not come from the course
materials, although the instruction says to answer only from them.

**Exercise 3.** The descriptions decide which tool is chosen. Rewrite the docstring of
`facts_by_name` so that it becomes the first choice for anything the question names, and
that of `search_course` so that it covers explanations and procedures only. Run this
part's cells again. Did the choices change, and did the answers?

---
## Part 5 — Step 4: a fixed query over the graph

A narrow tool: one Cypher query, written in advance, with one parameter. The model
supplies a name and the query does the rest. It runs through Week 6's `read`, in a
read-only transaction.

In [35]:
DOCUMENTS_MENTIONING = """
MATCH (d:Document)-[:HAS_CHUNK]->(:Chunk)-[:MENTIONS]->(e:Entity {name: $name})
RETURN DISTINCT d.name AS document ORDER BY document"""


@tool
def documents_mentioning(name: str) -> str:
    """List the course documents that mention a tool, topic or lab, given its name as
    the course graph spells it, for example 'Chroma'."""
    rows = read(driver, DOCUMENTS_MENTIONING, name=name)
    return "\n".join(r["document"] for r in rows) or f"No documents mention {name!r}."


for name in ("Chroma", "ChromaDB"):                     # no model involved
    print(f"{name}:\n{documents_mentioning.invoke({'name': name})}\n")

Chroma:
week-03-lecture-plan.md
week-03-lecture.md
week-03-practical-plan.md
week-03-practical.md

ChromaDB:
No documents mention 'ChromaDB'.



Both calls are well formed. The graph spells the tool *Chroma*, so *ChromaDB* finds
nothing, and the result reads like an answer: no document mentions it.

Give the agent the new tool beside the retrievers:

In [36]:
course_tools = [exchange_rate, calculate, facts_by_name, search_course, documents_mentioning]
agent = create_agent(model, tools=course_tools, system_prompt=COURSE,
                     middleware=[SEARCHES, LIMIT])
history = ask(agent, "Which course documents mention Chroma?")

user        Which course documents mention Chroma?
assistant   calls documents_mentioning {"name": "Chroma"}
tool        documents_mentioning → week-03-lecture-plan.md week-03-lecture.md week-03-practical-plan.md week-03-practical.md

──── answer ───────────────────────────────────────────────────────────
The following course documents mention **Chroma**:

- week-03-lecture-plan.md
- week-03-lecture.md
- week-03-practical-plan.md
- week-03-practical.md
───────────────────────────────────────────────────────────────────────
2 model calls in this turn, 42 since the kernel started


Which tool did the model choose? The fixed query answers this kind of question exactly;
from the retrievers, the model would have to infer the list from passages and facts.

**Exercise 4.** Ask `ask(agent, "Which course documents mention ChromaDB?")`. Does the
model report that none do, or try another name or another tool? How would you change
`documents_mentioning` so that a near miss is not silent?

---
## Part 6 — Step 5: memory as tools

In Week 7 the **application** wrote memory: one extraction call proposed facts, and the
code stored them. As tools, the **model** decides. `recall_memory` reads the store, and
`remember` writes to it: the agent's first **write** tool.

The store is Week 7's `MemoryStore`, with a collection of its own, `agent_memory`, which
is emptied each time the cell runs. Your Week 7 store is not touched.

In [37]:
from datetime import date
from memory import MemoryStore

store = MemoryStore(name="agent_memory", reset=True)


@tool
def recall_memory(query: str) -> str:
    """The stored facts about the user nearest in meaning to the query. Use it when an
    answer depends on the user: their project, their setup, what they did before."""
    facts = store.recall(query, k=3)
    return "\n".join(facts) if facts else "Nothing is stored about the user."


@tool
def remember(fact: str) -> str:
    """Store one fact about the user that will still matter in a later conversation,
    written as one short sentence about the user. Do not store the question itself or
    anything that matters only now."""
    [fact_id] = store.add([fact], source="agent", when=str(date.today()))
    return f"Stored as {fact_id}."


MEMORY = ("When the user's message tells you something about themselves that will matter "
          "in a later conversation, first store each such fact with remember. When an "
          "answer depends on the user, first recall what is stored. " + COURSE)

agent = create_agent(model, tools=course_tools + [recall_memory, remember],
                     system_prompt=MEMORY, middleware=[SEARCHES, LIMIT])
BACKGROUND = ("Some background before my questions: I missed the Week 4 lecture, so I "
              "have not set up vector search in Neo4j yet. For Lab 1 I used chunks of "
              "300 words.")
history = ask(agent, BACKGROUND)

user        Some background before my questions: I missed the Week 4 lecture, so I have not set up vector search in Neo4j yet. For Lab 1 I used chunks of 300 words.
assistant   calls remember {"fact": "The user missed the Week 4 lecture and has not set up vector search in Neo4j yet."}
assistant   calls remember {"fact": "For Lab 1, the user used chunks of 300 words."}
tool        remember → Stored as c40d87c9.
tool        remember → Stored as cd52637c.

──── answer ───────────────────────────────────────────────────────────
Got it. I've noted that you missed the Week 4 lecture (so vector search in Neo4j isn't set
up yet) and that you used 300-word chunks for Lab 1. What would you like to know?
───────────────────────────────────────────────────────────────────────
2 model calls in this turn, 44 since the kernel started


These are the facts of the message Week 7 extracted from, without its question. What did
the model store?

In [38]:
for f in store.all():
    print(f"{f['id']}  {f['when']}  {f['fact']}")

cd52637c  2026-10-25  For Lab 1, the user used chunks of 300 words.
c40d87c9  2026-10-25  The user missed the Week 4 lecture and has not set up vector search in Neo4j yet.


In Week 7 Part 3, an extraction call chose the facts from the same message, and the code
stored them. Here the model chose, and nothing checked its choice before it was written.

A new conversation: no `history`, so it starts empty.

In [39]:
history = ask(agent, "Have I set up vector search in Neo4j yet, and what chunk size did I use for Lab 1?")

user        Have I set up vector search in Neo4j yet, and what chunk size did I use for Lab 1?
assistant   calls recall_memory {"query": "vector search Neo4j setup"}
tool        recall_memory → The user missed the Week 4 lecture and has not set up vector search in Neo4j yet. For Lab 1, the user used chunks of 300 words.

──── answer ───────────────────────────────────────────────────────────
Based on what I recall:

- **Vector search in Neo4j**: You have **not set it up yet** (you missed the Week 4
lecture).
- **Lab 1 chunk size**: You used **300-word chunks**.
───────────────────────────────────────────────────────────────────────
2 model calls in this turn, 46 since the kernel started


Nothing of the first conversation was sent. Whatever the answer knows about you came from
the store, through `recall_memory`.

**Exercise 5.** Read each stored fact. Would your application have stored it, and which
should it not have stored? With a write tool, the store changes whenever the model
decides: how would you review, correct or undo what it wrote?

---
## Part 7 — Step 6: errors and a step limit

The lecture listed what can go wrong in a tool call. Here is how LangChain handles each by
default, and the two safeguards you add.

**Invalid arguments.** LangChain checks the arguments against the schema before it runs
the function. Called directly, the check raises an error:

In [40]:
try:
    calculate.invoke({"a": 1, "op": "^", "b": 2})
except Exception as error:
    print(type(error).__name__)
    print("\n".join(str(error).splitlines()[:3]))

ValidationError
1 validation error for calculate
op
  Input should be '+', '-', '*' or '/' [type=literal_error, input_value='^', input_type=str]


Inside an agent, LangChain returns this error to the model as the tool's result, and the
model can correct its call. That is the default for invalid arguments only.

**An exception inside the tool.** Any other error, raised while the function runs, ends
the run:

In [41]:
agent = create_agent(model, tools=[calculate], system_prompt=SYSTEM)
try:
    history = ask(agent, "Use the calculator: what is 7 divided by 0?")
except Exception as error:
    print("The run ended:", type(error).__name__, "|", error)

The run ended: ZeroDivisionError | division by zero


The model never saw the error, and the user received no answer. A tool that calls a
network service or a database fails in this way whenever the service is down.

Return such errors to the model instead, as the tool's result. LangChain's place for a
rule like this is **middleware**: code that runs around each model call or tool call,
without changing the loop.

In [42]:
from langchain.agents.middleware import ToolErrorMiddleware


def as_result(error, request):
    """The error, as the tool's result: the model reads it and can respond."""
    return f"Error: {type(error).__name__}: {error}"


agent = create_agent(model, tools=[calculate], system_prompt=SYSTEM,
                     middleware=[ToolErrorMiddleware(on_error=as_result)])
history = ask(agent, "Use the calculator: what is 7 divided by 0?")

user        Use the calculator: what is 7 divided by 0?
assistant   calls calculate {"op": "/", "a": 7, "b": 0}
tool        calculate → Error: ZeroDivisionError: division by zero

──── answer ───────────────────────────────────────────────────────────
Division by zero is undefined in mathematics. When you try to divide any number by zero
(including 7 ÷ 0), the operation has no meaningful result, which is why the calculator
returns a **ZeroDivisionError**.

In standard arithmetic, there is no number that, when multiplied by 0, gives 7 (or any
non-zero number), so the division cannot be defined.
───────────────────────────────────────────────────────────────────────
2 model calls in this turn, 49 since the kernel started


**A step limit.** `create_agent` sets no practical limit of its own: a model that keeps
calling tools may run for up to 9,999 steps, each one a request. Since Part 4 the course
agents have run with two limits. `ToolCallLimitMiddleware` counts the calls to one tool,
here `search_course`, and refuses those beyond the limit: the refusal is returned to the
model as the tool's result, and the model goes on without that tool.
`ModelCallLimitMiddleware` stops the whole run after a set number of model calls, whatever
the model does. The euro question of Part 3 needed three model calls, so a limit of two
shows what happens when that limit is reached:

In [43]:
agent = create_agent(model, tools=[exchange_rate, calculate], system_prompt=SYSTEM,
                     middleware=[ToolErrorMiddleware(on_error=as_result),
                                 ModelCallLimitMiddleware(run_limit=2, exit_behavior="end")])
history = ask(agent, "How many hryvnias is 250 euros at today's NBU rate?")

user        How many hryvnias is 250 euros at today's NBU rate?
assistant   calls exchange_rate {"currency": "EUR"}
tool        exchange_rate → 1 EUR = 50.5766 UAH, set by the NBU for 26.10.2026
assistant   calls calculate {"a": 250, "op": "*", "b": 50.5766}
tool        calculate → 12644.15

──── no answer ────────────────────────────────────────────────────────
The run was stopped at the limit on model calls before the model answered.
Model call limits exceeded: run limit (2/2)
───────────────────────────────────────────────────────────────────────
2 model calls in this turn, 51 since the kernel started


The run stopped at the limit without answering the question. A real limit is larger: set
it from the number of steps your questions need, with room to spare.

**The finished agent.** All the tools of this session, with the error handler and both
limits:

In [44]:
from agent_kit import COUNTER

agent = create_agent(model, tools=course_tools + [recall_memory, remember],
                     system_prompt=MEMORY,
                     middleware=[ToolErrorMiddleware(on_error=as_result), SEARCHES,
                                 ModelCallLimitMiddleware(run_limit=8, exit_behavior="end")])
print(f"model calls since the kernel started: {COUNTER.calls}")

model calls since the kernel started: 51


---
## Part 8 — Tools for your own project

**Exercise 6.** For your own project, write two or three tools as the lecture's table
describes them: a name, a description that says what the tool returns and when to use it,
and its parameters. Mark each as reading or writing. For each, state what a wrong but
well-formed call would cost: a wrong answer, or a wrong state that persists.

This list is the start of Lab 3, due at the end of Week 10: an agent with instructions, at
least two tools and persistent memory, reusing your RAG and Neo4j components. Week 9 asks
for the same tools as a specification.